## Serious Mental Illness (SMI) Diagnosis Trends 2008-2010

#### Programmer: Julia McGowan

#### Program Name: SMI_Inpatient_Claims

Background/Purpose: Use publicly available Medicare claims data from CMS to determine claims that meet Serious Mental Illness (SMI) definition using ICD-9 diagnosis codes.
Then analyze output dataset using SQL.

Note: Input data file is a sample of all 2008-2010 publicly available inpatient claims.

Data Source: https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf 

Variables referred to in this programming:
* SMI = serious mental illness
* IP = inpatient

In [0]:
# ===============
# Init PySpark
# ===============
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PySpark").getOrCreate()

In [0]:
# ================
# Program setup
# ================
from pyspark.sql.functions import col, lit
from pyspark.sql import functions as F
import pandas as pd
from pathlib import Path
from functools import reduce

# Demographic data variables to keep
demogkeep = ["DESYNPUF_ID", "SP_STATE_CODE", "YEAR", "BENE_BIRTH_DT", "BENE_SEX_IDENT_CD", "BENE_RACE_CD"]

# Inpatient claims variables to keep (incl. 10 ICD-9 diagnosis codes)
ipclmskeep = [
    "DESYNPUF_ID", "CLM_ID", "CLM_FROM_DT", "CLM_THRU_DT",] + [f"ICD9_DGNS_CD_{i}" for i in range(1, 11)]

# Sample data years 2008-2010
years = [2008, 2009, 2010]

# To read in demographic data dynamically:
def get_prefix(yr):
    if yr in [2008, 2009, 2010]:
        return f"DE1_0_{yr}"

# Limit records for testing
obs_limit = None

# SSA state code mapping
state_cd_map = {
"01":"AL","02":"AK","03":"AZ","04":"AR","05":"CA","06":"CO","07":"CT","08":"DE",
"09":"DC","10":"FL","11":"GA","12":"HI","13":"ID","14":"IL","15":"IN","16":"IA",
"17":"KS","18":"KY","19":"LA","20":"ME","21":"MD","22":"MA","23":"MI","24":"MN",
"25":"MS","26":"MO","27":"MT","28":"NE","29":"NV","30":"NH","31":"NJ","32":"NM",
"33":"NY","34":"NC","35":"ND","36":"OH","37":"OK","38":"OR","39":"PA","40":"RI",
"41":"SC","42":"SD","43":"TN","44":"TX","45":"UT","46":"VT","47":"VA","48":"WA",
"49":"WV","50":"WI","51":"WY"
}

# Sex category mapping
sex_cats = {
"1": "male",
"2": "female"
}

# Race category mapping
race_cats = {
"1": "white",
"2": "black",
"3": "other",
"4": "asian_pacific_islander",
"5": "hispanic",
"6": "north american native"
}

# SMI definition (ICD-9 codes)
smi_def = [

# schizophrenia
"29500","29501","29502","29503","29504","29505","29510","29511","29512","29513",
"29514","29515","29520","29521","29522","29523","29524","29525","29530","29531",
"29532","29533","29534","29535","29540","29541","29542","29543","29544","29545",
"29550","29551","29552","29553","29554","29555","29560","29561","29562","29563",
"29564","29565","29570","29571","29572","29573","29574","29575","29580","29581",
"29582","29583","29584","29585","29590","29591","29592","29593","29594","29595",

# bipolar disorders
"29600","29601","29602","29603","29604","29605","29606",
"29610","29611","29612","29613","29614","29615","29616",
"29640","29641","29642","29643","29644","29645","29646",
"29650","29651","29652","29653","29654","29655","29656",
"29660","29661","29662","29663","29664","29665","29666",
"2967","29680","29681","29682","29689",

# major depressive disorder
"29620","29621","29622","29623","29624","29625","29626",
"29630","29631","29632","29633","29634","29635","29636",

# paranoid disorders
"2970","2971","2972","2973","2978","2979",

# other psychoses
"2980","2981","2982","2983","2984","2988","2989"
]

In [0]:
# ===========================
# Read in demographic data
# ===========================
base = Path(r"dbfs:/Workspace/Users/juliamc@ad.unc.edu")

def read_demographic_data(yr):
    
	prefix = get_prefix(yr)
	file_name = f"{prefix}_Beneficiary_Summary_File_Sample_1.csv"
	path = str(base / file_name)
	year = yr
 
	df = spark.read.csv(
     path,
     header=True,
     inferSchema=True,
	)

	df = df.withColumn("YEAR", lit(year))

	if obs_limit:
		df = df.limit(obs_limit)

	# Keep only relevant columns
	df_cols = [c for c in demogkeep if c in df.columns]
	df = df[df_cols]

	return df

# Stack all years of demographic data together
dfs = [read_demographic_data(y) for y in years]
demographic_df = dfs[0]
for df in dfs[1:]:
    demographic_df = demographic_df.unionByName(df)

In [0]:
# ===========================
# Demographic data recodes
# ===========================
# Rename desynpuf_id & state_cd on input file
demographic_data_df = demographic_df.withColumnRenamed("SP_STATE_CODE","SSA_STATE_CD") \
.withColumnRenamed("DESYNPUF_ID", "patient_id")

# Create new state_cd variable and dummy variables for sex and race
demographic_data_df = demographic_data_df \
.withColumn(
    "state_cd",
    F.create_map([F.lit(x) for x in sum(state_cd_map.items(), ())])[F.col("SSA_STATE_CD")]
) \
.withColumn("race_white",  F.when(F.col("BENE_RACE_CD") == "1", 1).otherwise(0)) \
.withColumn("race_black",  F.when(F.col("BENE_RACE_CD") == "2", 1).otherwise(0)) \
.withColumn("race_other",  F.when(F.col("BENE_RACE_CD") == "3", 1).otherwise(0)) \
.withColumn("race_asian_pi", F.when(F.col("BENE_RACE_CD") == "4", 1).otherwise(0)) \
.withColumn("race_hispanic", F.when(F.col("BENE_RACE_CD") == "5", 1).otherwise(0)) \
.withColumn("race_native", F.when(F.col("BENE_RACE_CD") == "6", 1).otherwise(0)) \
.withColumn("female", F.when(F.col("BENE_SEX_IDENT_CD") == "2", 1).otherwise(0))

demographic_data_df = demographic_data_df.withColumn("BENE_BIRTH_DT", F.to_date(F.col("BENE_BIRTH_DT").cast("string"), "yyyMMdd"))

demographic_data_df.show()

+----------------+------------+----+-------------+-----------------+------------+--------+----------+----------+----------+-------------+-------------+-----------+------+
|      patient_id|SSA_STATE_CD|YEAR|BENE_BIRTH_DT|BENE_SEX_IDENT_CD|BENE_RACE_CD|state_cd|race_white|race_black|race_other|race_asian_pi|race_hispanic|race_native|female|
+----------------+------------+----+-------------+-----------------+------------+--------+----------+----------+----------+-------------+-------------+-----------+------+
|00013D2EFD8E45D1|          26|2008|   1923-05-01|                1|           1|      MO|         1|         0|         0|            0|            0|          0|     0|
|00016F745862898F|          39|2008|   1943-01-01|                1|           1|      PA|         1|         0|         0|            0|            0|          0|     0|
|0001FDD721E223DC|          39|2008|   1936-09-01|                2|           1|      PA|         1|         0|         0|            0|        

In [0]:
# ================================
# Read in inpatient claims data
# ================================
def read_clms_data(start_year, end_year):

    file_name = f"DE1_0_{start_year}_to_{end_year}_Inpatient_Claims_Sample_1.csv"
    path = str(base / file_name)

    df = spark.read.csv(
        path,
        header=True,
        inferSchema=True,
    )

    if obs_limit:
        df = df.head(obs_limit)

    df_cols = [c for c in ipclmskeep if c in df.columns]
    df = df[df_cols]

    return df

ip_clms_df = read_clms_data(min(years), max(years))

# Rename columns
ip_clms_df = ip_clms_df.withColumnRenamed("DESYNPUF_ID", "patient_id") \
    .withColumnRenamed("CLM_FROM_DT", "start_dt") \
    .withColumnRenamed("CLM_THRU_DT", "end_dt")

# Format start_dt and end_dt as Date types
ip_clms_df = ip_clms_df.withColumn("end_dt", F.to_date(F.col("end_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("start_dt", F.to_date(F.col("start_dt").cast("string"), "yyyyMMdd"))

# Rename diagnosis columns
for i in range(1, 11):
    ip_clms_df = ip_clms_df.withColumnRenamed(f"ICD9_DGNS_CD_{i}", f"dx{i}")

ip_clms_df.show()

+----------------+---------------+----------+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+
|      patient_id|         CLM_ID|  start_dt|    end_dt|  dx1|  dx2|  dx3|  dx4|  dx5|  dx6|  dx7|  dx8|  dx9|dx10|
+----------------+---------------+----------+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+
|00013D2EFD8E45D1|196661176988405|2010-03-12|2010-03-13| 7802|78820|V4501| 4280| 2720| 4019|V4502|73300|E9330|NULL|
|00016F745862898F|196201177000368|2009-04-12|2009-04-18| 1970| 4019| 5853| 7843| 2768|71590| 2724|19889| 5849|NULL|
|00016F745862898F|196661177015632|2009-08-31|2009-09-02| 6186| 2948|56400| NULL| NULL| NULL| NULL| NULL| NULL|NULL|
|00016F745862898F|196091176981058|2009-09-17|2009-09-20|29623|30390|71690|34590|V1581|32723| NULL| NULL| NULL|NULL|
|00016F745862898F|196261176983265|2010-06-26|2010-07-01| 3569| 4019| 3542|V8801|78820| 2639| 7840| 7856| 4271|NULL|
|00052705243EA128|196991176971757|2008-09-12|2008-09-12|  486| 3004|4273

In [0]:
# ==============================================================================================================
# From Medicare inpatient claims data -- identify beneficiaries with SMI ICD code in any diagnosis (Dx1-Dx10)
# ==============================================================================================================
dx_cols = [f"dx{i}" for i in range(1,11)]

# Look for SMI Dx in Dx codes (Dx1-Dx10)
smi_cond = reduce(lambda a,b: a | b, [F.col(c).isin(smi_def) for c in dx_cols])

# Finder file of beneficiaries that meet either definition
finder_df = (
    ip_clms_df
    .withColumn("smi", F.when(smi_cond,1).otherwise(0))
    .filter((F.col("smi")==1))
    .select("patient_id","smi")
    .distinct()
)

In [0]:
# =================================================================================================
# Inner join IP claims with finder file (on patient_id) to reduce data to population of interest   
# Keep variables of interest
# Drop Dx1-Dx10 after creating outcome flags (no longer needed)
# =================================================================================================
# To keep claims of beneficiaries with an SMI Dx at some point during study period:
ip_clms_df = (
    ip_clms_df
    .join(finder_df.select("patient_id"), on="patient_id", how="inner")
)

# Create SMI flag on claims-level data
# Look for SMI Dx in Dx codes (Dx1-Dx10)
smi_cond = reduce(lambda a,b: a | b, [F.col(c).isin(smi_def) for c in dx_cols])

# Create year for joining on demographic data
ip_clms_df = ip_clms_df.withColumn("dt_string", F.col("end_dt").cast("string")) \
.withColumn("YEAR", F.substring(F.col("dt_string"), 1, 4)) \
.drop("dt_string")

# Add SMI flag to claims data
ip_clms_df = (
    ip_clms_df
    .withColumn("smi", F.when(smi_cond,1).otherwise(0))
)

ip_clms_df.show()

+----------------+---------------+----------+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+---+
|      patient_id|         CLM_ID|  start_dt|    end_dt|  dx1|  dx2|  dx3|  dx4|  dx5|  dx6|  dx7|  dx8|  dx9| dx10|YEAR|smi|
+----------------+---------------+----------+----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+---+
|00016F745862898F|196201177000368|2009-04-12|2009-04-18| 1970| 4019| 5853| 7843| 2768|71590| 2724|19889| 5849| NULL|2009|  0|
|00016F745862898F|196661177015632|2009-08-31|2009-09-02| 6186| 2948|56400| NULL| NULL| NULL| NULL| NULL| NULL| NULL|2009|  0|
|00016F745862898F|196091176981058|2009-09-17|2009-09-20|29623|30390|71690|34590|V1581|32723| NULL| NULL| NULL| NULL|2009|  1|
|00016F745862898F|196261176983265|2010-06-26|2010-07-01| 3569| 4019| 3542|V8801|78820| 2639| 7840| 7856| 4271| NULL|2010|  0|
|00108066CA1FACCE|196501176978540|2009-09-30|2009-10-07|29623| 2875|27651| 7245|60000| 7220|53081| 4019|30523| NULL|20

In [0]:
# =================================================================
# Left join demographic data onto IP claims to finalize SMI flag
# Join on patient_id, year 
# Data is at the patient_id, clm_id level
# =================================================================
ip_clms_with_demog_df = ip_clms_df.join(
    demographic_data_df,
    on=["patient_id", "YEAR"],
    how="left"
)

# Calculate age based on end_dt and BENE_BIRTH_DT
ip_clms_with_demog_df = ip_clms_with_demog_df.withColumn(
    "age", F.floor(F.months_between(F.col("end_dt"), F.col("BENE_BIRTH_DT")) / 12)
)

# Apply age restrictions to SMI (age >=18)
ip_clms_with_demog_df = (
    ip_clms_with_demog_df
    .withColumn("smi",
        F.when((F.col("age") >= 18) & (F.col("smi") == 1), 1).otherwise(0)
    )
)

# Drop extra variables
ip_clms_with_demog_df = ip_clms_with_demog_df.drop("SSA_STATE_CD", "BENE_BIRTH_DT")

for i in range(1,11):
    ip_clms_with_demog_df = ip_clms_with_demog_df.drop(f"dx{i}")

ip_clms_with_demog_df.show()

+----------------+----+---------------+----------+----------+---+-----------------+------------+--------+----------+----------+----------+-------------+-------------+-----------+------+---+
|      patient_id|YEAR|         CLM_ID|  start_dt|    end_dt|smi|BENE_SEX_IDENT_CD|BENE_RACE_CD|state_cd|race_white|race_black|race_other|race_asian_pi|race_hispanic|race_native|female|age|
+----------------+----+---------------+----------+----------+---+-----------------+------------+--------+----------+----------+----------+-------------+-------------+-----------+------+---+
|0099BA9C4EC79141|2010|196451177013223|2010-03-18|2010-03-19|  1|                2|           1|      VA|         1|         0|         0|            0|            0|          0|     1| 29|
|0016D2185D29BC11|2008|196631176972622|2008-04-20|2008-04-23|  0|                2|           1|      PA|         1|         0|         0|            0|            0|          0|     1| 81|
|004E88D8CE2C3FDB|2009|196591176961926|2009-10-25|

In [0]:
# =======================================================================================================================
# Create output dataset by collapsing data to a patient_id, year level (taking max of SMI flag for a given year)
# If any claim for a person-year has a flag value of 1 then the final flag has a value of 1 a the patient_id, year level
# =======================================================================================================================
output_df = (
    ip_clms_with_demog_df
    .groupBy("patient_id", "YEAR")
    .agg(
        F.max("smi").alias("smi"),
        F.first("state_cd").alias("state_cd"),
        F.first("race_white").alias("race_white"),
        F.first("race_black").alias("race_black"),
        F.first("race_other").alias("race_other"),
        F.first("race_asian_pi").alias("race_asian_pi"),
        F.first("race_hispanic").alias("race_hispanic"),
        F.first("race_native").alias("race_native"),
        F.first("female").alias("female"),
        F.first("age").alias("age")
    )
)

# Add age categories
output_df = (
    output_df
    .withColumn("age_0_17", F.when((F.col("age") >= 0) & (F.col("age") <= 17), 1).otherwise(0))
    .withColumn("age_18_25", F.when((F.col("age") >= 19) & (F.col("age") <= 25), 1).otherwise(0))
    .withColumn("age_26_34", F.when((F.col("age") >= 26) & (F.col("age") <= 34), 1).otherwise(0))
    .withColumn("age_35_49", F.when((F.col("age") >= 35) & (F.col("age") <= 49), 1).otherwise(0))
    .withColumn("age_50_64", F.when((F.col("age") >= 50) & (F.col("age") <= 64), 1).otherwise(0))
    .withColumn("age_65_plus", F.when(F.col("age") >= 65, 1).otherwise(0))
)

# Order output dataset
output_df = output_df.orderBy("patient_id", "state_cd", "year")
output_df.show()

# Create temp view for SQL
output_df.createOrReplaceTempView("output_df")

+----------------+----+---+--------+----------+----------+----------+-------------+-------------+-----------+------+---+--------+---------+---------+---------+---------+-----------+
|      patient_id|YEAR|smi|state_cd|race_white|race_black|race_other|race_asian_pi|race_hispanic|race_native|female|age|age_0_17|age_18_25|age_26_34|age_35_49|age_50_64|age_65_plus|
+----------------+----+---+--------+----------+----------+----------+-------------+-------------+-----------+------+---+--------+---------+---------+---------+---------+-----------+
|00016F745862898F|2009|  1|      PA|         1|         0|         0|            0|            0|          0|     0| 66|       0|        0|        0|        0|        0|          1|
|00016F745862898F|2010|  0|      PA|         1|         0|         0|            0|            0|          0|     0| 67|       0|        0|        0|        0|        0|          1|
|00108066CA1FACCE|2009|  1|    NULL|         1|         0|         0|            0|       

In [0]:
%sql
-- Count of Medicare beneficiaries meeting SMI definition by state and year

SELECT distinct state_cd, year, SUM(smi) AS smi_count
FROM output_df where state_cd IN ('NC', 'SC', 'VA', 'NY', 'GA')
GROUP BY state_cd, year
ORDER BY state_cd, year;

state_cd,year,smi_count
GA,2008,34
GA,2009,43
GA,2010,26
NC,2008,59
NC,2009,59
NC,2010,37
NY,2008,105
NY,2009,98
NY,2010,59
SC,2008,5


In [0]:
%sql
-- Count of Medicare beneficiaries meeting SMI definition by race and year
SELECT
    year,
    SUM(race_white)      AS white,
    SUM(race_black)      AS black,
    SUM(race_other)      AS other,
    SUM(race_asian_pi)   AS asian_pi,
    SUM(race_hispanic)   AS hispanic,
    SUM(race_native)     AS native_am
FROM output_df where year is not null
GROUP BY year
ORDER BY year;

year,white,black,other,asian_pi,hispanic,native_am
2008,1947,255,78,0,47,0
2009,1927,252,79,0,44,0
2010,1101,162,37,0,31,0


In [0]:
%sql
-- Count of Medicare beneficiaries meeting SMI definition by sex and year
SELECT
    year,
    CASE 
        WHEN female = 1 THEN 'Female'
        ELSE 'Male'
    END AS sex,
    SUM(CASE WHEN smi = 1 THEN 1 ELSE 0 END) AS smi_count
FROM output_df where year is not NULL
GROUP BY year, sex
ORDER BY year, sex;

year,sex,smi_count
2008,Female,927
2008,Male,730
2009,Female,951
2009,Male,692
2010,Female,499
2010,Male,426


In [0]:
%sql
-- Count of Medicare beneficiaries meeting SMI definition by race (Black and white) and sex across all years in the dataset
SELECT
    year,

    SUM(CASE WHEN female = 1 AND race_white = 1 AND smi = 1 THEN 1 ELSE 0 END) AS white_female_smi,
    SUM(CASE WHEN female = 0 AND race_white = 1 AND smi = 1 THEN 1 ELSE 0 END) AS white_male_smi,

    SUM(CASE WHEN female = 1 AND race_black = 1 AND smi = 1 THEN 1 ELSE 0 END) AS black_female_smi,
    SUM(CASE WHEN female = 0 AND race_black = 1 AND smi = 1 THEN 1 ELSE 0 END) AS black_male_smi

FROM output_df
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year;

year,white_female_smi,white_male_smi,black_female_smi,black_male_smi
2008,781,608,95,84
2009,794,578,100,76
2010,412,351,61,51
